## Extractive Summarizer: SBERT Fine-Tuning + K-Means
Mở Google Colab Notebook: [tại đây](https://colab.research.google.com/drive/15b_4azZOfc5DGCwXuzOLkh0s9KVaKaVB)

Notebook này thực hiện:
1. **Fine-Tuning SBERT (Tiếng Anh & Tiếng Việt)** sử dụng tập cặp câu Oracle và `CosineSimilarityLoss`
3. **Đánh giá & Ablation Study**: So sánh Lead-3, TextRank, SBERT-No-KMeans và FineTuned-SBERT-KMeans trên các chỉ số ROUGE-1, ROUGE-2, ROUGE-L và Diversity Score.

In [1]:
import torch
is_gpu = torch.cuda.is_available()
print(f"CUDA/GPU Available: {is_gpu}")
if is_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Tên GPU: {gpu_name}")
    print(f"Tổng dung lượng VRAM: {vram_gb:.2f} GB")

CUDA/GPU Available: True
Tên GPU: Tesla T4
Tổng dung lượng VRAM: 14.56 GB


## 1. Cài đặt Môi trường & Thư viện

In [2]:
import os
import sys

if 'colab' in str(get_ipython()):
    if not os.path.exists('Extractive-Summarizer'):
        !git clone https://github.com/kttt294/Extractive-Summarizer.git
    if os.path.exists('Extractive-Summarizer') and not os.getcwd().endswith('Extractive-Summarizer'):
        %cd Extractive-Summarizer
    !git pull
    !pip install -q -r requirements.txt
sys.path.append(os.getcwd())

Cloning into 'Extractive-Summarizer'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (155/155), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 155 (delta 67), reused 118 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (155/155), 14.22 MiB | 20.66 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/Extractive-Summarizer
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 55.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 99.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## 2. Fine-Tuning Mô hình SBERT Song ngữ (English & Vietnamese)

**Cấu hình Siêu tham số:**
- `sample_data_count = 3000`: Lấy 3,000 bài báo (~12,000 cặp câu Oracle). Đạt điểm bão hòa hiệu năng cao nhất, tránh hiện tượng *Catastrophic Forgetting* và tối ưu hóa thời gian chạy trên Colab GPU T4.
- `epochs = 3`: Chuẩn mực tối ưu từ tác giả Sentence-Transformers để Fine-tune Bi-Encoder qua CosineSimilarityLoss.
- `batch_size = 32`: Tối ưu hiệu năng tính toán ma trận trên GPU NVIDIA T4.

In [3]:
from src.train import train_finetune_sbert

# Fine-tuning Mô hình SBERT Tiếng Anh (CNN/DailyMail)
print("BẮT ĐẦU FINE-TUNE SBERT TIẾNG ANH (3,000 BÀI BÁO)")
model_en_path = train_finetune_sbert(lang='en', epochs=3, batch_size=32, sample_data_count=3000)

# Fine-tuning Mô hình SBERT Tiếng Việt (VietNews)
print("\n\nBẮT ĐẦU FINE-TUNE SBERT TIẾNG VIỆT (3,000 BÀI BÁO)")
model_vi_path = train_finetune_sbert(lang='vi', epochs=3, batch_size=32, sample_data_count=3000)

BẮT ĐẦU FINE-TUNE SBERT TIẾNG ANH (3,000 BÀI BÁO)
Nạp mô hình SBERT Gốc: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Đã nạp thành công 3000 bài báo Tinh hoa VI [train] qua lọc phân tầng.
Đang tạo các cặp câu Oracle cho Fine-Tuning (mục tiêu: 12000 cặp).
Đã khởi tạo thành công 12001 cặp câu huấn luyện Oracle.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.029846
1000,0.003541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-Tuning hoàn tất. Mô hình đã được lưu tại: /content/Extractive-Summarizer/models/finetuned_sbert_vi



In [6]:
import shutil
shutil.make_archive('finetuned_sbert_en', 'zip', './models/finetuned_sbert_en')
shutil.make_archive('finetuned_sbert_vi', 'zip', './models/finetuned_sbert_vi')
if 'colab' in str(get_ipython()):
    from google.colab import files
    files.download('finetuned_sbert_en.zip')
    files.download('finetuned_sbert_vi.zip')
    print("Đã nén và tải weights về máy.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đã nén và tải weights về máy.


## 3. Khung Đánh giá Thực nghiệm & Ablation Study
So sánh trực tiếp kết quả ROUGE-1, ROUGE-2, ROUGE-L và Diversity giữa 4 phương pháp:
1. **Lead-3 Baseline**
2. **TextRank Baseline**
3. **SBERT-No-KMeans** *(Direct Top-K - Ablation Study)*
4. **FineTuned-SBERT-KMeans** *(Mô hình Đề xuất)*

In [21]:
# Chạy Đánh giá & Ablation Study trên 200 bài báo thử nghiệm

from src.evaluate import evaluate_framework
evaluate_framework(lang='en', sample_count=200)
evaluate_framework(lang='vi', sample_count=200)

Đánh giá với ngôn ngữ EN - Số lượng: 200)

Mô hình                    | Silhouette | Diversity  | Compress (%) | ROUGE-1 (%)  | ROUGE-2 (%)  | ROUGE-L (%)  | BERTScore 
Lead-3                     | 0.0000     | 0.0000     | 19.54        | 31.0237      | 12.7243      | 21.2789      | 0.4395    
TextRank                   | 0.0000     | 0.0000     | 26.71        | 23.2290      | 7.3552       | 15.9873      | 0.3455    
Pretrained-SBERT-KMeans    | 0.0743     | 0.7817     | 33.55        | 24.6801      | 9.1687       | 16.5708      | 0.4239    
SBERT-No-KMeans            | 0.0000     | 0.7966     | 38.42        | 22.5682      | 8.8523       | 15.6044      | 0.4364    
FineTuned-SBERT-KMeans     | 0.0495     | 0.9326     | 34.80        | 23.9970      | 8.9069       | 16.0737      | 0.4218    


Đánh giá với ngôn ngữ VI - Số lượng: 200)

Mô hình                    | Silhouette | Diversity  | Compress (%) | ROUGE-1 (%)  | ROUGE-2 (%)  | ROUGE-L (%)  | BERTScore 
Lead-3                     | 0

## 4. Demo Tóm tắt Văn bản Trực tiếp
Thử nghiệm trực tiếp pipeline tóm tắt trên một văn bản báo chí thực tế.

In [20]:
from src.preprocess import resolve_language
from src.evaluate import run_sbert_pipeline

sample_article = '''
Trí tuệ nhân tạo (AI) đang tạo ra một cuộc cách mạng công nghệ sâu sắc, làm thay đổi toàn diện phương thức vận hành của các ngành công nghiệp trên toàn cầu với tốc độ nhanh chóng chưa từng có trong lịch sử.
Những bước tiến vượt bậc gần đây trong lĩnh vực Xử lý Ngôn ngữ Tự nhiên (NLP) đã cho phép các hệ thống máy tính có khả năng phân tích, hiểu sâu ngữ cảnh và tóm tắt những tài liệu văn bản phức tạp chỉ trong vài giây.
Trong số các phương pháp hiện đại, kỹ thuật tóm tắt trích xuất (Extractive Summarization) nổi bật nhờ khả năng lựa chọn trực tiếp các câu chứa đựng nhiều thông tin cốt lõi nhất từ văn bản gốc mà không làm biến đổi ý nghĩa ban đầu.
Bằng cách tận dụng các mô hình vector nhúng tiên tiến như Sentence-BERT (SBERT), hệ thống máy tính có thể bắt trọn các mối quan hệ ngữ nghĩa tinh tế và độ tương đồng giữa các câu trong toàn bộ bài viết.
Tiếp đó, việc ứng dụng thuật toán phân cụm K-Means giúp gom nhóm hiệu quả các khái niệm tương đồng thành các cụm chủ đề con, đảm bảo bản tóm tắt thu được bao quát đa dạng nhiều góc độ thông tin khác nhau.
Sau bước phân cụm, kỹ thuật lọc trùng lặp sau xử lý (Post-filtering) sử dụng ngưỡng Cosine Similarity để loại bỏ triệt để các câu có nội dung trùng dư thừa, tạo ra một bản tóm tắt mượt mà, ngắn gọn và giàu giá trị cho người đọc.
Phương pháp tiếp cận nối tiếp 2 giai đoạn này đại diện cho một giải pháp tối ưu, mở ra triển vọng ứng dụng rộng lớn cho các hệ thống tóm tắt văn bản tự động chất lượng cao trong kỷ nguyên số.
'''

lang = resolve_language(sample_article, user_choice='auto')
summary_text, summary_sents, sil_score, div_score = run_sbert_pipeline(sample_article, lang=lang, use_finetuned=True)

print(f"Ngôn ngữ tự động nhận diện: {lang.upper()}")
print(f"Chỉ số Nội tại (Intrinsic): Silhouette Score = {sil_score:.4f} | Diversity Score = {div_score:.4f}")
print("\n--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---")
for i, sent in enumerate(summary_sents, 1):
    print(f"{i}. {sent}")

Ngôn ngữ tự động nhận diện: VI
Chỉ số Nội tại (Intrinsic): Silhouette Score = 0.1563 | Diversity Score = 0.0026

--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---
1. Trí tuệ nhân tạo (AI) đang tạo ra một cuộc cách mạng công nghệ sâu sắc, làm thay đổi toàn diện phương thức vận hành của các ngành công nghiệp trên toàn cầu với tốc độ nhanh chóng chưa từng có trong lịch sử.
2. Những bước tiến vượt bậc gần đây trong lĩnh vực Xử lý Ngôn ngữ Tự nhiên (NLP) đã cho phép các hệ thống máy tính có khả năng phân tích, hiểu sâu ngữ cảnh và tóm tắt những tài liệu văn bản phức tạp chỉ trong vài giây.
3. Trong số các phương pháp hiện đại, kỹ thuật tóm tắt trích xuất (Extractive Summarization) nổi bật nhờ khả năng lựa chọn trực tiếp các câu chứa đựng nhiều thông tin cốt lõi nhất từ văn bản gốc mà không làm biến đổi ý nghĩa ban đầu.


In [19]:
sample_article_2 = """
Thí sinh đăng ký nguyện vọng xét tuyển đại học, cao đẳng năm 2026 từ 2/7 đến 17 giờ 14/7
Thứ Năm, 02/07/2026 07:32
 |
Tuyển sinh
Báo Tin Tức trên Google News
Ngay sau khi biết điểm thi tốt nghiệp Trung học phổ thông, bắt đầu từ hôm nay (2/7) đến 17 giờ ngày 14/7, thí sinh thực hiện đăng ký, điều chỉnh, bổ sung nguyện vọng xét tuyển đại học, cao đẳng trực tuyến trên Hệ thống hỗ trợ tuyển sinh chung của Bộ Giáo dục và Đào tạo hoặc Cổng dịch vụ công quốc gia.
Nhiều điều chỉnh trong phương thức xét tuyển đại học
Thí sinh đăng ký nguyện vọng xét tuyển đại học năm 2026 từ ngày 2/7-14/7
Đa dạng phương thức xét tuyển đại học 2026
Chia sẻ Facebook
Chia sẻ Zalo
Chú thích ảnh
Ảnh minh họa: Nguyễn Lành/TTXVN
Năm nay, thí sinh được đăng kí tối đa 15 nguyện vọng và sắp xếp theo thứ tự ưu tiên từ cao xuống thấp; đối với các thí sinh đăng kí xét tuyển vào các ngành đào tạo giáo viên, các cơ sở đào tạo chỉ xét tuyển các nguyện vọng từ 1 đến 5.

Theo quy định, toàn bộ nguyện vọng xét tuyển của thí sinh đều phải đăng ký trên Hệ thống để xử lý chung, kể cả trường hợp thí sinh đã tham gia xét tuyển theo các phương thức riêng của cơ sở đào tạo (nếu có). Việc xử lý nguyện vọng tập trung nhằm bảo đảm nguyên tắc mỗi thí sinh chỉ trúng tuyển một nguyện vọng cao nhất, góp phần nâng cao tính công bằng, minh bạch trong tuyển sinh.

Lưu ý, các thí sinh đăng ký xét tuyển vào các chương trình đào tạo giáo viên, chương trình đào tạo thuộc lĩnh vực sức khỏe có cấp Giấy phép hành nghề, chương trình đào tạo thuộc lĩnh vực pháp luật căn cứ vào ngưỡng bảo đảm chất lượng đầu vào do các cơ sở đào tạo công bố để điều chỉnh nguyện vọng (các cơ sở đào tạo cập nhật ngưỡng đầu vào trước 17 giờ ngày 10/7/2026).

Từ ngày 15/7 đến 17 giờ ngày 21/7/2026, thí sinh nộp lệ phí xét tuyển trực tuyến theo số lượng nguyện vọng đã đăng kí.

Thí sinh cần tìm hiểu kỹ tài liệu hướng dẫn và phải thực hiện đúng, đủ, hết quy trình đăng ký xét tuyển. Thí sinh chưa rõ các nội dung khai báo, nộp lệ phí xét tuyển, có thể liên hệ với cán bộ tại các điểm tiếp nhận hoặc cán bộ của cơ sở đào tạo trực các số điện thoại hỗ trợ công tác tuyển sinh để được hướng dẫn.

Kết quả điểm trúng tuyển và danh sách thí sinh trúng tuyển đợt 1 sẽ được các cơ sở đào tạo công bố trước 17 giờ ngày 13/8/2026.

Sau khi có kết quả trúng tuyển, tất cả thí sinh, kể cả thí sinh trúng tuyển thẳng, phải xác nhận nhập học trực tuyến trước 17 giờ ngày 21/8/2026 nếu có nguyện vọng theo học. Đây là bước bắt buộc để hoàn tất quá trình trúng tuyển.
"""

lang = resolve_language(sample_article_2, user_choice='auto')
summary_text, summary_sents, sil_score, div_score = run_sbert_pipeline(sample_article_2, lang=lang, use_finetuned=True)

print(f"Ngôn ngữ tự động nhận diện: {lang.upper()}")
print(f"Chỉ số Nội tại (Intrinsic): Silhouette Score = {sil_score:.4f} | Diversity Score = {div_score:.4f}")
print(f"\n--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---")
for i, sent in enumerate(summary_sents, 1):
    print(f"{i}. {sent}")

Ngôn ngữ tự động nhận diện: VI
Chỉ số Nội tại (Intrinsic): Silhouette Score = 0.0885 | Diversity Score = 0.0086

--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---
1. Theo quy định, toàn bộ nguyện vọng xét tuyển của thí sinh đều phải đăng ký trên Hệ thống để xử lý chung, kể cả trường hợp thí sinh đã tham gia xét tuyển theo các phương thức riêng của cơ sở đào tạo (nếu có).
2. Từ ngày 15/7 đến 17 giờ ngày 21/7/2026, thí sinh nộp lệ phí xét tuyển trực tuyến theo số lượng nguyện vọng đã đăng kí.
3. Thí sinh cần tìm hiểu kỹ tài liệu hướng dẫn và phải thực hiện đúng, đủ, hết quy trình đăng ký xét tuyển.
